# Monthly Data Update & Model Refresh

Run once per month to pull the latest data, re-engineer features, retrain the model, and refresh all outputs.

## 1. Configuration

The target month defaults to the most recently completed calendar month.
Precipitation is fetched from **Open-Meteo** (no token needed).
311 data is fetched from **NYC Open Data** — optionally add an app token to raise the rate limit.

| Token | Required? | Where to get it |
|-------|-----------|----------------|
| `NYC_APP_TOKEN` | Optional | https://data.cityofnewyork.us/profile/app_tokens |


In [8]:
import os, requests, calendar, warnings, json
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta
from shapely.geometry import box, Point
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = r'C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f'

# ── API tokens ────────────────────────────────────────────────────────────────
# Precipitation uses Open-Meteo — no token needed.
# NYC_APP_TOKEN is optional but raises the Socrata rate limit (1000 → 10,000 req/hr).
# Get one free at: https://data.cityofnewyork.us/profile/app_tokens
NYC_APP_TOKEN = ''

# ── Target month (default = last completed month) ─────────────────────────────
_today      = date.today()
_last_month = _today.replace(day=1) - relativedelta(months=1)
TARGET_YEAR  = _last_month.year
TARGET_MONTH = _last_month.month
TARGET_YM    = f'{TARGET_YEAR}-{TARGET_MONTH:02d}'

# Override here if you want a specific month, e.g.:
# TARGET_YEAR, TARGET_MONTH, TARGET_YM = 2025, 12, '2025-12'

START_DATE = f'{TARGET_YEAR}-{TARGET_MONTH:02d}-01'
END_DATE   = f'{TARGET_YEAR}-{TARGET_MONTH:02d}-{calendar.monthrange(TARGET_YEAR, TARGET_MONTH)[1]:02d}'

print(f'Target month : {TARGET_YM}')
print(f'Date range   : {START_DATE}  →  {END_DATE}')
print(f'NOAA token   : {"SET" if NOAA_TOKEN != "YOUR_NOAA_CDO_TOKEN" else "NOT SET — update before running"}')


Target month : 2026-02
Date range   : 2026-02-01  →  2026-02-28
NOAA token   : NOT SET — update before running


## 2. Fetch New Data

### 2a. Central Park Precipitation — Open-Meteo Archive API

Uses the [Open-Meteo Archive API](https://open-meteo.com/en/docs/historical-weather-api) — **no API key required**.
Coordinates are Central Park (40.7829 N, 73.9654 W), precipitation in inches to match existing data.


In [9]:
OPENMETEO_URL = 'https://archive-api.open-meteo.com/v1/archive'
OPENMETEO_PARAMS = {
    'latitude':          40.7829,
    'longitude':         -73.9654,
    'start_date':        START_DATE,
    'end_date':          END_DATE,
    'daily':             'precipitation_sum',
    'timezone':          'America/New_York',
    'precipitation_unit':'inch',
}

resp = requests.get(OPENMETEO_URL, params=OPENMETEO_PARAMS, timeout=30)
resp.raise_for_status()
om_data = resp.json()

times = om_data['daily']['time']
prcp_vals = om_data['daily']['precipitation_sum']

rows = []
for dt, val in zip(times, prcp_vals):
    val = float(val) if val is not None else 0.0
    rows.append({
        'STATION':    'USW00094728',
        'NAME':       'NY CITY CENTRAL PARK, NY US',
        'DATE':       dt,
        'PRCP':       round(val, 4),
        'Year':       int(dt[:4]),
        'Month':      int(dt[5:7]),
        'Year_Month': TARGET_YM,
    })

new_prcp = pd.DataFrame(rows)
print(f'Fetched {len(new_prcp)} daily records for {TARGET_YM}')
print(f'Total monthly precipitation : {new_prcp["PRCP"].sum():.2f} inches')
print(new_prcp.head(3).to_string(index=False))


Fetched 28 daily records for 2026-02
Total monthly precipitation : 2.04 inches
    STATION                        NAME       DATE  PRCP  Year  Month Year_Month
USW00094728 NY CITY CENTRAL PARK, NY US 2026-02-01   0.0  2026      2    2026-02
USW00094728 NY CITY CENTRAL PARK, NY US 2026-02-02   0.0  2026      2    2026-02
USW00094728 NY CITY CENTRAL PARK, NY US 2026-02-03   0.0  2026      2    2026-02


### 2b. NYC 311 — Cave-in / Depression / Sunken Sidewalk Reports

In [10]:
SOCRATA_311 = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

# Column rename map: Socrata API field → cleaned CSV column name
RENAME_311 = {
    'unique_key':               'Unique Key',
    'created_date':             'Created Date',
    'complaint_type':           'Problem (formerly Complaint Type)',
    'descriptor':               'Problem Detail (formerly Descriptor)',
    'borough':                  'Borough',
    'latitude':                 'Latitude',
    'longitude':                'Longitude',
    'location':                 'Location',
    'x_coordinate_state_plane_': 'X Coordinate (State Plane)',
    'y_coordinate_state_plane_': 'Y Coordinate (State Plane)',
}

# Sinkhole-related descriptors (all variants used in the dataset)
CAVE_IN_TYPES = (
    "complaint_type='Street Condition' AND "
    "descriptor IN('Cave-in','Depression','Sidewalk Collapsed','Sidewalk Sunken','Sewer Cave-in')"
)

def _socrata_fetch(url, where_clause, app_token='', limit=5000):
    """Fetch all matching rows from a Socrata endpoint, handling pagination."""
    headers = {'X-App-Token': app_token} if app_token else {}
    all_rows = []
    offset = 0
    while True:
        params = {'$where': where_clause,
                  '$limit': limit,
                  '$offset': offset,
                  '$order': 'created_date DESC'}
        r = requests.get(url, params=params, headers=headers, timeout=60)
        r.raise_for_status()
        batch = r.json()
        all_rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return all_rows

cave_in_where = (
    f"complaint_type='Street Condition' AND "
    f"descriptor IN('Cave-in','Depression','Sidewalk Collapsed','Sidewalk Sunken','Sewer Cave-in') AND "
    f"created_date >= '{START_DATE}T00:00:00' AND "
    f"created_date <= '{END_DATE}T23:59:59' AND "
    f"latitude IS NOT NULL"
)

raw_311 = _socrata_fetch(SOCRATA_311, cave_in_where, app_token=NYC_APP_TOKEN)
new_311  = pd.DataFrame(raw_311).rename(columns=RENAME_311)

# Ensure numeric types and add date columns
new_311['Latitude']  = pd.to_numeric(new_311.get('Latitude',  pd.Series(dtype=float)), errors='coerce')
new_311['Longitude'] = pd.to_numeric(new_311.get('Longitude', pd.Series(dtype=float)), errors='coerce')
new_311['Unique Key'] = pd.to_numeric(new_311['Unique Key'], errors='coerce')
new_311['year']       = TARGET_YEAR
new_311['month']      = TARGET_MONTH
new_311['year_month'] = TARGET_YM

# Keep only columns that exist in the cleaned CSV (drop any extras)
keep_cols = ['Unique Key','Created Date','Problem (formerly Complaint Type)',
             'Problem Detail (formerly Descriptor)','Borough',
             'Latitude','Longitude','Location',
             'X Coordinate (State Plane)','Y Coordinate (State Plane)',
             'year','month','year_month']
new_311 = new_311[[c for c in keep_cols if c in new_311.columns]]
new_311 = new_311.dropna(subset=['Latitude','Longitude'])

print(f'Fetched {len(new_311)} sinkhole-related 311 records for {TARGET_YM}')
if len(new_311):
    print(new_311['Problem Detail (formerly Descriptor)'].value_counts().to_string())


Fetched 697 sinkhole-related 311 records for 2026-02
Problem Detail (formerly Descriptor)
Cave-in    697


### 2c. NYC 311 — Water Main Breaks

In [11]:
water_where = (
    f"complaint_type='Water System' AND "
    f"descriptor LIKE '%Water Main Break%' AND "
    f"created_date >= '{START_DATE}T00:00:00' AND "
    f"created_date <= '{END_DATE}T23:59:59' AND "
    f"latitude IS NOT NULL"
)

RENAME_WATER = {
    'unique_key':               'Unique Key',
    'created_date':             'Created Date',
    'borough':                  'Borough',
    'x_coordinate_state_plane_': 'X Coordinate (State Plane)',
    'y_coordinate_state_plane_': 'Y Coordinate (State Plane)',
    'latitude':                 'Latitude',
    'longitude':                'Longitude',
    'location':                 'Location',
    'complaint_type':           'Problem (formerly Complaint Type)',
    'descriptor':               'Problem Detail (formerly Descriptor)',
}

raw_water = _socrata_fetch(SOCRATA_311, water_where, app_token=NYC_APP_TOKEN)
new_water  = pd.DataFrame(raw_water).rename(columns=RENAME_WATER)

new_water['Latitude']    = pd.to_numeric(new_water.get('Latitude',  pd.Series(dtype=float)), errors='coerce')
new_water['Longitude']   = pd.to_numeric(new_water.get('Longitude', pd.Series(dtype=float)), errors='coerce')
new_water['Unique Key']  = pd.to_numeric(new_water['Unique Key'], errors='coerce')
new_water['Year']        = TARGET_YEAR
new_water['Month']       = TARGET_MONTH
new_water['Year_Month']  = TARGET_YM

water_keep = ['Unique Key','Created Date','Borough',
              'X Coordinate (State Plane)','Y Coordinate (State Plane)',
              'Latitude','Longitude','Location',
              'Problem (formerly Complaint Type)',
              'Problem Detail (formerly Descriptor)',
              'Year','Month','Year_Month']
new_water = new_water[[c for c in water_keep if c in new_water.columns]]
new_water = new_water.dropna(subset=['Latitude','Longitude'])

print(f'Fetched {len(new_water)} water main break records for {TARGET_YM}')


Fetched 1343 water main break records for 2026-02


## 3. Append to Cleaned CSVs

In [12]:
def _append_dedup(path, new_df, dedup_col, label):
    """Load existing CSV, append new rows, deduplicate, save."""
    existing = pd.read_csv(path)
    before   = len(existing)
    combined = pd.concat([existing, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=[dedup_col]).reset_index(drop=True)
    combined.to_csv(path, index=False)
    added = len(combined) - before
    print(f'{label}: {before:,} → {len(combined):,} rows  (+{added} new)')
    return combined

# Precipitation: dedup on DATE (one record per day)
prcp_path = os.path.join(BASE, 'cleaned_precipitation.csv')
_append_dedup(prcp_path, new_prcp, dedup_col='DATE', label='Precipitation')

# 311 cave-ins: dedup on Unique Key
csv_311_path = os.path.join(BASE, 'cleaned_311_points.csv')
_append_dedup(csv_311_path, new_311, dedup_col='Unique Key', label='311 cave-ins')

# Water breaks: dedup on Unique Key
water_path = os.path.join(BASE, 'cleaned_water.csv')
_append_dedup(water_path, new_water, dedup_col='Unique Key', label='Water breaks')

print('\nCleaned CSVs updated.')


Precipitation: 5,784 → 5,812 rows  (+28 new)
311 cave-ins: 49,191 → 49,888 rows  (+697 new)
Water breaks: 20,757 → 22,100 rows  (+1343 new)

Cleaned CSVs updated.


## 4. Re-run Feature Engineering

Recomputes all features from the updated cleaned CSVs. Rolling-window features are recalculated on the full history to ensure correctness at the boundary of the new month.

In [13]:
# ── Build 500 m NYC grid (same as featureEngineering.ipynb) ──────────────────
nyc = gpd.read_file(r'C:\Users\szaha\Downloads\Borough_Boundaries_20260328.geojson').to_crs(epsg=32618)
nyc_union = nyc.dissolve().reset_index(drop=True)

xmin, ymin, xmax, ymax = nyc_union.total_bounds
cell_size = 500
grid_cells = [box(x, y, x+cell_size, y+cell_size)
              for x in np.arange(xmin, xmax, cell_size)
              for y in np.arange(ymin, ymax, cell_size)]
grid = gpd.GeoDataFrame(geometry=grid_cells, crs=nyc_union.crs)
grid_clipped = gpd.overlay(grid, nyc_union, how='intersection').reset_index(drop=True)
grid_clipped['grid_id'] = range(len(grid_clipped))

# Build neighbor lookup (1 m buffer)
grid_buf = grid_clipped[['grid_id','geometry']].copy()
grid_buf['geometry'] = grid_buf.geometry.buffer(1)
neighbor_join = gpd.sjoin(
    grid_buf.rename(columns={'grid_id':'neighbor_id'}),
    grid_clipped[['grid_id','geometry']],
    how='inner', predicate='intersects'
)
neighbors = (neighbor_join[neighbor_join['neighbor_id'] != neighbor_join['grid_id']]
             [['grid_id','neighbor_id']].reset_index(drop=True))

print(f'Grid: {len(grid_clipped):,} cells  |  neighbor pairs: {len(neighbors):,}')


Grid: 3,718 cells  |  neighbor pairs: 26,334


In [14]:
# ── Feature set 1: 311 complaints ────────────────────────────────────────────
df_311_raw = pd.read_csv(os.path.join(BASE, 'cleaned_311_points.csv'))
df_311_raw = df_311_raw.dropna(subset=['Latitude','Longitude']).copy()

gdf_311 = gpd.GeoDataFrame(
    df_311_raw,
    geometry=gpd.points_from_xy(df_311_raw['Longitude'], df_311_raw['Latitude']),
    crs='EPSG:4326'
).to_crs(epsg=32618)

joined_311 = gpd.sjoin(gdf_311, grid_clipped[['grid_id','geometry']],
                       how='inner', predicate='within')[['grid_id','year_month']].copy()
joined_311['year_month'] = joined_311['year_month'].astype(str)

# Full panel: all grid × all months
all_months = sorted(df_311_raw['year_month'].dropna().unique())
panel = pd.MultiIndex.from_product(
    [grid_clipped['grid_id'], all_months], names=['grid_id','year_month']
).to_frame(index=False)

monthly_311 = joined_311.groupby(['grid_id','year_month']).size().reset_index(name='complaint_count')
feat_311 = panel.merge(monthly_311, on=['grid_id','year_month'], how='left')
feat_311['complaint_count'] = feat_311['complaint_count'].fillna(0).astype(int)
feat_311['month'] = feat_311['year_month'].str[5:7].astype(int)

# Rolling features with shift(1) to prevent leakage
feat_311 = feat_311.sort_values(['grid_id','year_month']).reset_index(drop=True)
feat_311['prior_month_complaints']    = feat_311.groupby('grid_id')['complaint_count'].shift(1).fillna(0)
feat_311['complaints_in_last_3months']= feat_311.groupby('grid_id')['complaint_count'].shift(1).rolling(3, min_periods=1).sum().values

# Spatial lag: neighboring sinkhole incidents (prior month)
monthly_311_agg = feat_311[['grid_id','year_month','complaint_count']].copy()
monthly_311_agg['lag_complaints'] = feat_311.groupby('grid_id')['complaint_count'].shift(1).fillna(0).values
neighbor_311 = neighbors.merge(monthly_311_agg[['grid_id','year_month','lag_complaints']].rename(columns={'grid_id':'neighbor_id','lag_complaints':'n_complaints'}), on='neighbor_id', how='left')
neighbor_sinkhole = neighbor_311.groupby(['grid_id','year_month'])['n_complaints'].sum().reset_index(name='neighboring_sinkhole_incidents')
feat_311 = feat_311.merge(neighbor_sinkhole, on=['grid_id','year_month'], how='left')
feat_311['neighboring_sinkhole_incidents'] = feat_311['neighboring_sinkhole_incidents'].fillna(0)

print(f'311 features: {feat_311.shape}  |  months: {feat_311["year_month"].nunique()}')


311 features: (267696, 7)  |  months: 72


In [15]:
# ── Feature set 2: Water main breaks ─────────────────────────────────────────
df_water_raw = pd.read_csv(os.path.join(BASE, 'cleaned_water.csv'))
df_water_raw = df_water_raw.dropna(subset=['Latitude','Longitude']).copy()

gdf_water = gpd.GeoDataFrame(
    df_water_raw,
    geometry=gpd.points_from_xy(df_water_raw['Longitude'], df_water_raw['Latitude']),
    crs='EPSG:4326'
).to_crs(epsg=32618)

joined_water = gpd.sjoin(gdf_water, grid_clipped[['grid_id','geometry']],
                         how='inner', predicate='within')[['grid_id','Year_Month']].copy()
joined_water.columns = ['grid_id','year_month']

panel_w = pd.MultiIndex.from_product(
    [grid_clipped['grid_id'], all_months], names=['grid_id','year_month']
).to_frame(index=False)

monthly_water = joined_water.groupby(['grid_id','year_month']).size().reset_index(name='break_count')
feat_water = panel_w.merge(monthly_water, on=['grid_id','year_month'], how='left')
feat_water['break_count'] = feat_water['break_count'].fillna(0).astype(int)
feat_water = feat_water.sort_values(['grid_id','year_month']).reset_index(drop=True)

feat_water['breaks_last_month']   = feat_water.groupby('grid_id')['break_count'].shift(1).fillna(0)
feat_water['breaks_last_3_months']= feat_water.groupby('grid_id')['break_count'].shift(1).rolling(3, min_periods=1).sum().values

# Spatial lag: neighboring water breaks (prior month)
feat_water['lag_breaks'] = feat_water.groupby('grid_id')['break_count'].shift(1).fillna(0).values
neighbor_w = neighbors.merge(feat_water[['grid_id','year_month','lag_breaks']].rename(columns={'grid_id':'neighbor_id','lag_breaks':'n_breaks'}), on='neighbor_id', how='left')
neighbor_water_agg = neighbor_w.groupby(['grid_id','year_month'])['n_breaks'].sum().reset_index(name='neighboring_water_breaks')
feat_311 = feat_311.merge(neighbor_water_agg, on=['grid_id','year_month'], how='left')
feat_311['neighboring_water_breaks'] = feat_311['neighboring_water_breaks'].fillna(0)

# Drop temporary column before saving water features
feat_water = feat_water.drop(columns=['lag_breaks'])

print(f'Water features: {feat_water.shape}')
print(f'311 features now include neighbor_water_breaks: {feat_311.shape}')


Water features: (267696, 5)
311 features now include neighbor_water_breaks: (267696, 8)


In [16]:
# ── Feature set 3: Precipitation ─────────────────────────────────────────────
df_prcp_raw = pd.read_csv(os.path.join(BASE, 'cleaned_precipitation.csv'))

# Aggregate to monthly FIRST on the full history (prevents rolling window edge effects)
monthly_prcp = df_prcp_raw.groupby('Year_Month')['PRCP'].sum().reset_index(name='monthly_rain')
monthly_prcp = monthly_prcp.sort_values('Year_Month').reset_index(drop=True)
monthly_prcp['rain_last_3_months'] = monthly_prcp['monthly_rain'].shift(1).rolling(3, min_periods=1).sum().values
monthly_prcp = monthly_prcp[monthly_prcp['Year_Month'].isin(all_months)].reset_index(drop=True)

# Broadcast to every grid cell (precipitation is city-wide from one station)
feat_prcp = pd.MultiIndex.from_product(
    [grid_clipped['grid_id'], monthly_prcp['Year_Month']], names=['grid_id','year_month']
).to_frame(index=False)
feat_prcp = feat_prcp.merge(monthly_prcp.rename(columns={'Year_Month':'year_month'}),
                             on='year_month', how='left')

print(f'Precipitation features: {feat_prcp.shape}')
print(f'Latest month in prcp: {feat_prcp["year_month"].max()}')


Precipitation features: (267696, 4)
Latest month in prcp: 2026-02


In [17]:
# ── Feature set 4: Pavement ratings (forward-fill, no monthly fetch needed) ──
# Pavement is already forward-filled per cell. For the new month, carry forward
# each cell's most recent known rating (same logic as original feature engineering).

df_pave_existing = pd.read_csv(os.path.join(BASE, 'features_pavement_grid.csv'))
last_pave = df_pave_existing.sort_values('year_month').groupby('grid_id').last()[['pavement_rating']].reset_index()

new_pave_rows = last_pave.copy()
new_pave_rows['year_month'] = TARGET_YM
feat_pave_new = pd.concat([df_pave_existing, new_pave_rows], ignore_index=True)
feat_pave_new = feat_pave_new.drop_duplicates(subset=['grid_id','year_month']).sort_values(['grid_id','year_month']).reset_index(drop=True)

print(f'Pavement features: {feat_pave_new.shape}  |  new month rows: {len(new_pave_rows):,}')


Pavement features: (260260, 3)  |  new month rows: 3,718


In [18]:
# ── Save updated feature CSVs ─────────────────────────────────────────────────
feat_311.to_csv(os.path.join(BASE, 'features_311_grid.csv'),           index=False)
feat_water.to_csv(os.path.join(BASE, 'features_water_grid.csv'),       index=False)
feat_prcp.to_csv(os.path.join(BASE, 'features_precipitation_grid.csv'),index=False)
feat_pave_new.to_csv(os.path.join(BASE, 'features_pavement_grid.csv'), index=False)
# Bedrock is static — no update needed

print('Feature CSVs saved:')
for name, df in [('features_311_grid',   feat_311),
                 ('features_water_grid',  feat_water),
                 ('features_precip_grid', feat_prcp),
                 ('features_pave_grid',   feat_pave_new)]:
    print(f'  {name}: {df.shape}  last month = {df["year_month"].max()}')


Feature CSVs saved:
  features_311_grid: (267696, 8)  last month = 2026-02
  features_water_grid: (267696, 5)  last month = 2026-02
  features_precip_grid: (267696, 4)  last month = 2026-02
  features_pave_grid: (260260, 3)  last month = 2026-02


## 5. Retrain Model & Generate Predictions

In [19]:
FEATURE_COLUMNS = [
    'month','prior_month_complaints','complaints_in_last_3months',
    'breaks_last_month','breaks_last_3_months',
    'monthly_rain','rain_last_3_months',
    'pavement_rating','elev_ft','depth_ft',
    'neighboring_sinkhole_incidents','neighboring_water_breaks',
]

# Merge all features
df_bed = pd.read_csv(os.path.join(BASE, 'features_bedrock_grid.csv'))

df = feat_311.copy()
df = df.merge(feat_water[['grid_id','year_month','breaks_last_month','breaks_last_3_months']],
              on=['grid_id','year_month'], how='left')
df = df.merge(feat_prcp[['grid_id','year_month','monthly_rain','rain_last_3_months']],
              on=['grid_id','year_month'], how='left')
df = df.merge(feat_pave_new[['grid_id','year_month','pavement_rating']],
              on=['grid_id','year_month'], how='left')
df = df.merge(df_bed[['grid_id','elev_ft','depth_ft']], on='grid_id', how='left')

df['has_sinkhole'] = (df['complaint_count'] > 0).astype(int)
df = df[df['year_month'] >= '2020-04'].reset_index(drop=True)
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Dataset: {df.shape}  (dropped {before-len(df)} rows with nulls)')
print(f'Date range: {df["year_month"].min()} → {df["year_month"].max()}')


Dataset: (249106, 16)  (dropped 7436 rows with nulls)
Date range: 2020-04 → 2026-02


In [20]:
# Temporal train/test split (same cutoff as original)
cutoff = '2024-01'
train  = df[df['year_month'] <  cutoff]
test   = df[df['year_month'] >= cutoff]

X_train, y_train = train[FEATURE_COLUMNS], train['has_sinkhole']
X_test,  y_test  = test [FEATURE_COLUMNS], test ['has_sinkhole']

smote = SMOTE(random_state=42, sampling_strategy=0.2)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=12,
    min_samples_leaf=10, min_samples_split=20,
    random_state=42, n_jobs=-1
)
rf.fit(X_train_res, y_train_res)

# Tune threshold
y_proba = rf.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, y_proba)
fb = (1 + 0.25) * (prec * rec) / (0.25 * prec + rec + 1e-9)
best_threshold = thr[np.argmax(fb)]

print(f'Model retrained on {len(X_train_res):,} samples (SMOTE)')
print(f'Optimal threshold : {best_threshold:.4f}')


Model retrained on 170,706 samples (SMOTE)
Optimal threshold : 0.3182


In [21]:
# Predictions for latest (new) month
latest_month = df['year_month'].max()
df_latest    = df[df['year_month'] == latest_month].copy()
df_latest['sinkhole_probability'] = rf.predict_proba(df_latest[FEATURE_COLUMNS])[:, 1]
df_latest['prediction']           = (df_latest['sinkhole_probability'] >= best_threshold).astype(int)
df_latest['prediction_label']     = df_latest['prediction'].map({0:'No Sinkhole Expected', 1:'Sinkhole Risk'})
df_latest = df_latest.sort_values('sinkhole_probability', ascending=False).reset_index(drop=True)

pred_path = os.path.join(BASE, 'predictions_latest_month.csv')
df_latest[['grid_id','year_month','sinkhole_probability','prediction_label']].to_csv(pred_path, index=False)

print(f'Predictions for : {latest_month}')
print(f'At-risk cells   : {df_latest["prediction"].sum():,} / {len(df_latest):,}')
print(f'Saved to        : {pred_path}')


Predictions for : 2026-02
At-risk cells   : 168 / 3,718
Saved to        : C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\predictions_latest_month.csv


## 6. Refresh Outputs

Regenerates the interactive risk map and the static dashboard PNG.

In [22]:
import json as json_lib, plotly.express as px

# Rebuild grid geometry for map
grid_pred = grid_clipped[['grid_id','geometry']].merge(
    df_latest[['grid_id','sinkhole_probability','prediction']], on='grid_id', how='left'
)
grid_pred['prediction']           = grid_pred['prediction'].fillna(0).astype(int)
grid_pred['sinkhole_probability'] = grid_pred['sinkhole_probability'].fillna(0).round(4)

grid_pred_wgs = grid_pred.copy().to_crs(epsg=4326)
grid_pred_wgs['prediction_label'] = grid_pred_wgs['prediction'].map({0:'No Sinkhole Expected', 1:'Sinkhole Risk'})

geojson = json_lib.loads(grid_pred_wgs.to_json())
for feat in geojson['features']:
    feat['id'] = feat['properties']['grid_id']

fig_map = px.choropleth_mapbox(
    grid_pred_wgs, geojson=geojson, locations='grid_id',
    color='sinkhole_probability', color_continuous_scale='YlOrRd',
    range_color=[0, 1], mapbox_style='carto-positron',
    zoom=10, center={'lat': 40.7128, 'lon': -74.0060}, opacity=0.75,
    hover_name='grid_id',
    hover_data={'grid_id': False, 'sinkhole_probability': ':.4f', 'prediction_label': True},
    labels={'sinkhole_probability': 'Probability', 'prediction_label': 'Prediction'},
    title=f'NYC Sinkhole Risk Prediction — {latest_month}',
)
fig_map.update_layout(
    margin={'r':0,'t':50,'l':0,'b':0},
    coloraxis_colorbar=dict(title='Sinkhole<br>Probability',
                            tickvals=[0,.25,.5,.75,1],
                            ticktext=['0%','25%','50%','75%','100%']),
)
map_path = os.path.join(BASE, 'sinkhole_risk_map.html')
fig_map.write_html(map_path, include_plotlyjs='cdn', full_html=True)
print(f'Risk map saved: {map_path}')


Risk map saved: C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\sinkhole_risk_map.html


In [23]:
# ── Static dashboard PNG (matplotlib) ────────────────────────────────────────
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Borough assignment for latest month
boro_col = [c for c in nyc.columns if 'boro' in c.lower() and 'name' in c.lower()][0]
grid_with_boro = (
    gpd.sjoin(grid_clipped[['grid_id','geometry']], nyc[['geometry',boro_col]],
              how='left', predicate='intersects')
    .drop_duplicates('grid_id')[['grid_id',boro_col]]
    .rename(columns={boro_col:'borough'})
)
latest_with_boro = df_latest[['grid_id','sinkhole_probability','prediction']].merge(grid_with_boro, on='grid_id', how='left')
boro_stats = (latest_with_boro.groupby('borough')
              .agg(avg_risk=('sinkhole_probability','mean'), n_at_risk=('prediction','sum'), n_cells=('grid_id','count'))
              .reset_index().sort_values('avg_risk', ascending=False))
boro_stats['pct_at_risk'] = (boro_stats['n_at_risk'] / boro_stats['n_cells'] * 100).round(1)

# Monthly trend
monthly_rows = []
for ym in sorted(df['year_month'].unique()):
    df_m = df[df['year_month']==ym]; proba = rf.predict_proba(df_m[FEATURE_COLUMNS])[:,1]; pred=(proba>=best_threshold).astype(int)
    monthly_rows.append({'year_month':ym,'pct_at_risk':pred.mean()*100,'n_at_risk':int(pred.sum())})
monthly_df = pd.DataFrame(monthly_rows)
monthly_df['period'] = np.where(monthly_df['year_month']<'2024-01','Train','Test')

# Location labels from cache
cache_path = os.path.join(BASE, 'grid_locations_cache.csv')
if os.path.exists(cache_path):
    cache_df = pd.read_csv(cache_path); cache_df['grid_id'] = cache_df['grid_id'].astype(int)
    df_latest = df_latest.merge(cache_df[['grid_id','location_label']], on='grid_id', how='left')
df_latest['location_label'] = df_latest.get('location_label', pd.Series(dtype=str)).fillna('Cell '+df_latest['grid_id'].astype(str))

total_cells=len(df_latest); n_at_risk_now=int(df_latest['prediction'].sum())
pct_at_risk=n_at_risk_now/total_cells*100; avg_prob=df_latest['sinkhole_probability'].mean(); top1=df_latest.iloc[0]
top10=df_latest.head(10).copy(); top10['label']=top10['location_label']

fig_png, axes = plt.subplots(3, 2, figsize=(18, 16))
fig_png.patch.set_facecolor('#f8f9fa')

for ax in axes[0]:
    ax.set_axis_off(); ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_linewidth(1.5); spine.set_edgecolor('#dee2e6')

delta = pct_at_risk - monthly_df.iloc[-2]['pct_at_risk']
delta_str = f'{"▲" if delta>=0 else "▼"} {abs(delta):.1f}% vs prior month'
delta_col = '#e74c3c' if delta >= 0 else '#27ae60'
axes[0,0].text(0.5,0.62,f'{pct_at_risk:.1f}%',ha='center',va='center',fontsize=46,fontweight='bold',color='#e74c3c',transform=axes[0,0].transAxes)
axes[0,0].text(0.5,0.88,'% of NYC at High Risk',ha='center',va='center',fontsize=13,fontweight='bold',color='#2c3e50',transform=axes[0,0].transAxes)
axes[0,0].text(0.5,0.72,f'{n_at_risk_now:,} of {total_cells:,} cells  |  {latest_month}',ha='center',va='center',fontsize=10,color='#7f8c8d',transform=axes[0,0].transAxes)
axes[0,0].text(0.5,0.30,delta_str,ha='center',va='center',fontsize=11,color=delta_col,transform=axes[0,0].transAxes)
axes[0,1].text(0.5,0.62,f'{avg_prob*100:.2f}%',ha='center',va='center',fontsize=46,fontweight='bold',color='#e67e22',transform=axes[0,1].transAxes)
axes[0,1].text(0.5,0.88,'City-Wide Avg Sinkhole Probability',ha='center',va='center',fontsize=13,fontweight='bold',color='#2c3e50',transform=axes[0,1].transAxes)
axes[0,1].text(0.5,0.35,f'Highest-risk: {top1["location_label"]}\n({top1["sinkhole_probability"]:.1%})',ha='center',va='center',fontsize=10,color='#7f8c8d',transform=axes[0,1].transAxes)

ax_top10=axes[1,0]; ax_top10.set_facecolor('white')
probs=top10['sinkhole_probability'].values*100; labels=top10['label'].values
colors=plt.cm.YlOrRd(top10['sinkhole_probability'].values)
bars=ax_top10.barh(range(len(labels)),probs,color=colors,edgecolor='white',height=0.7)
ax_top10.set_yticks(range(len(labels))); ax_top10.set_yticklabels(labels,fontsize=9)
ax_top10.set_xlabel('Sinkhole Probability (%)',fontsize=10)
ax_top10.set_title(f'Top 10 Highest-Risk Locations ({latest_month})',fontsize=11,fontweight='bold',pad=8)
ax_top10.invert_yaxis()
for i,(bar,prob) in enumerate(zip(bars,probs)):
    ax_top10.text(prob+0.3,i,f'{prob:.1f}%',va='center',fontsize=8.5,color='#333')
ax_top10.spines['top'].set_visible(False); ax_top10.spines['right'].set_visible(False)
ax_top10.set_xlim(0,max(probs)*1.2)

ax_trend=axes[1,1]; ax_trend.set_facecolor('white')
train_sub=monthly_df[monthly_df['period']=='Train']; test_sub=monthly_df[monthly_df['period']=='Test']
ax_trend.plot(range(len(train_sub)),train_sub['pct_at_risk'].values,color='#3498db',linewidth=2,linestyle=':',marker='o',markersize=3,label='Train')
ax_trend.plot(range(len(train_sub),len(monthly_df)),test_sub['pct_at_risk'].values,color='#e74c3c',linewidth=2,marker='o',markersize=3,label='Test')
split_x=len(train_sub)-0.5
ax_trend.axvline(split_x,color='grey',linestyle='--',linewidth=1.5,alpha=0.8)
ax_trend.text(split_x+0.3,ax_trend.get_ylim()[1]*0.95,'Train/Test split',fontsize=8,color='grey',va='top')
all_months_list=monthly_df['year_month'].tolist(); tick_pos=list(range(0,len(all_months_list),6))
ax_trend.set_xticks(tick_pos); ax_trend.set_xticklabels([all_months_list[i] for i in tick_pos],rotation=45,fontsize=8)
ax_trend.set_ylabel('% of Cells at High Risk',fontsize=10)
ax_trend.set_title('Monthly % of Cells at High Risk',fontsize=11,fontweight='bold',pad=8)
ax_trend.legend(fontsize=9); ax_trend.grid(axis='y',alpha=0.3)
ax_trend.spines['top'].set_visible(False); ax_trend.spines['right'].set_visible(False)

ax_bavg=axes[2,0]; ax_bavg.set_facecolor('white')
bsorted=boro_stats.sort_values('avg_risk')
bcolors=plt.cm.Oranges(bsorted['avg_risk'].values/bsorted['avg_risk'].max())
bbars=ax_bavg.barh(bsorted['borough'],bsorted['avg_risk']*100,color=bcolors,edgecolor='white',height=0.6)
for bar,val in zip(bbars,bsorted['avg_risk']*100):
    ax_bavg.text(val+0.01,bar.get_y()+bar.get_height()/2,f'{val:.2f}%',va='center',fontsize=9)
ax_bavg.set_xlabel('Avg Probability (%)',fontsize=10)
ax_bavg.set_title('Average Risk Probability by Borough',fontsize=11,fontweight='bold',pad=8)
ax_bavg.spines['top'].set_visible(False); ax_bavg.spines['right'].set_visible(False)
ax_bavg.set_xlim(0,bsorted['avg_risk'].max()*100*1.35)

ax_bct=axes[2,1]; ax_bct.set_facecolor('white')
nsorted=boro_stats.sort_values('n_at_risk')
nmax = max(nsorted['n_at_risk'].max(), 1)
ncolors=plt.cm.Reds(nsorted['n_at_risk'].values/nmax)
nbars=ax_bct.barh(nsorted['borough'],nsorted['n_at_risk'],color=ncolors,edgecolor='white',height=0.6)
for bar,row in zip(nbars,nsorted.itertuples()):
    ax_bct.text(row.n_at_risk+0.2,bar.get_y()+bar.get_height()/2,f'{int(row.n_at_risk)} ({row.pct_at_risk:.0f}%)',va='center',fontsize=9)
ax_bct.set_xlabel('Number of High-Risk Cells',fontsize=10)
ax_bct.set_title('High-Risk Cell Count by Borough',fontsize=11,fontweight='bold',pad=8)
ax_bct.spines['top'].set_visible(False); ax_bct.spines['right'].set_visible(False)
ax_bct.set_xlim(0,nmax*1.35)

fig_png.suptitle(f'NYC Sinkhole Risk Dashboard  —  {latest_month}',fontsize=18,fontweight='bold',color='#2c3e50',y=1.01)
plt.tight_layout(pad=2.0)
out_png = os.path.join(BASE, 'sinkhole_dashboard.png')
fig_png.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.close()
print(f'Dashboard PNG saved: {out_png}')
print(f'\nUpdate complete for {latest_month}.')


Dashboard PNG saved: C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\sinkhole_dashboard.png

Update complete for 2026-02.
